In [74]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

In [75]:
ls

 Volume in drive C has no label.
 Volume Serial Number is 8E0E-0AB3

 Directory of C:\Users\User\Downloads\Python-Finance-QuantConnect\06-Financial-Concepts-with-Python

23/04/2026  09:54    <DIR>          .
25/03/2026  08:24    <DIR>          ..
23/04/2026  09:36    <DIR>          .ipynb_checkpoints
23/04/2026  09:07            28,930 01-Sharpe-Ratios.ipynb
23/04/2026  08:57           722,286 02-MPT-Portfolio-Optimization.ipynb
23/04/2026  09:24           274,548 03-CAPM-Capital-Asset-Pricing-Model.ipynb
25/03/2026  08:24            90,236 AAPL.csv
25/03/2026  08:24           298,511 amazon_2010.csv
25/03/2026  08:24           155,241 apple.csv
25/03/2026  08:24           325,467 apple_2010.csv
25/03/2026  08:24            93,128 COST.csv
25/03/2026  08:24            90,762 DG.csv
25/03/2026  08:24            94,325 FB.csv
25/03/2026  08:24           319,995 GE_2010.csv
25/03/2026  08:24           154,859 msft.csv
23/04/2026  09:54            24,320 Python in Finance.ipynb
25/03/2026 

In [76]:
aapl= pd.read_csv('apple.csv', index_col='Date')
msft= pd.read_csv('msft.csv', index_col='Date')

In [77]:
aapl.head()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2015-12-31,26.752501,26.757500,26.205000,26.315001,24.266081,163649200
2016-01-04,25.652500,26.342501,25.500000,26.337500,24.286833,270597600
2016-01-05,26.437500,26.462500,25.602501,25.677500,23.678219,223164000
2016-01-06,25.139999,25.592501,24.967501,25.174999,23.214844,273829600
2016-01-07,24.670000,25.032499,24.107500,24.112499,22.235069,324377600


In [78]:
msft.head()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2015-12-31,56.040001,56.189999,55.419998,55.480000,50.298279,27334100
2016-01-04,54.320000,54.799999,53.389999,54.799999,49.681782,53778000
2016-01-05,54.930000,55.389999,54.540001,55.049999,49.908432,34079700
2016-01-06,54.320000,54.400002,53.639999,54.049999,49.001839,39518900
2016-01-07,52.700001,53.490002,52.070000,52.169998,47.297417,56564900


In [79]:
aapl.columns

Index(['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'], dtype='object')

In [80]:
aapl['Daily Returns']= aapl['Adj Close'].pct_change(1)
msft['Daily Returns']= msft['Adj Close'].pct_change(1)

In [81]:
aapl= aapl.dropna()
msft= msft.dropna()

In [82]:
aapl.head()

,Open,High,Low,Close,Adj Close,Volume,Daily Returns
Date,,,,,,,
2016-01-04,25.652500,26.342501,25.500000,26.337500,24.286833,270597600,0.000855
2016-01-05,26.437500,26.462500,25.602501,25.677500,23.678219,223164000,-0.025059
2016-01-06,25.139999,25.592501,24.967501,25.174999,23.214844,273829600,-0.019570
2016-01-07,24.670000,25.032499,24.107500,24.112499,22.235069,324377600,-0.042205
2016-01-08,24.637501,24.777500,24.190001,24.240000,22.352642,283192000,0.005288


In [83]:
msft.head()

,Open,High,Low,Close,Adj Close,Volume,Daily Returns
Date,,,,,,,
2016-01-04,54.320000,54.799999,53.389999,54.799999,49.681782,53778000,-0.012257
2016-01-05,54.930000,55.389999,54.540001,55.049999,49.908432,34079700,0.004562
2016-01-06,54.320000,54.400002,53.639999,54.049999,49.001839,39518900,-0.018165
2016-01-07,52.700001,53.490002,52.070000,52.169998,47.297417,56564900,-0.034783
2016-01-08,52.369999,53.279999,52.150002,52.330002,47.442482,48754000,0.003067


In [84]:
aapl['Daily Returns'].std()

0.01870000923066472

In [85]:
msft['Daily Returns'].std()

0.01701449290723031

In [86]:
def compute_sharpe_ratio(data,risk_free_rate=0):
    mean_return = data['Daily Returns'].mean()
    std = data['Daily Returns'].std()
    sharpe_ratio = (mean_return-risk_free_rate) / std
    return sharpe_ratio *(252**0.5)
# to calculate sharpe ratio

In [87]:
compute_sharpe_ratio(aapl)

np.float64(1.229522590246574)

In [88]:
compute_sharpe_ratio(msft)

np.float64(1.3052460572259035)

In [89]:
def compute_sortino_ratio(data, target=0, risk_free_rate=0):
    mean_return = data['Daily Returns'].mean()
    downside = data[data['Daily Returns'] < target]['Daily Returns']
    std = downside.std()
    sortino_ratio = (mean_return-risk_free_rate) / std
    return sortino_ratio * (252**0.5)
# to calcute sortino ratio


In [91]:
compute_sortino_ratio(aapl)

np.float64(1.6135822434562253)

In [92]:
import scipy.stats
from scipy.stats import norm

In [101]:
def compute_prob_sharpe_ratio(data, benchmark=0):
    
    sr = compute_sharpe_ratio(data, 0)
    
    skew = scipy.stats.skew(data['Daily Returns'])
    # Use fisher kurtosis
    
    kurtosis = scipy.stats.kurtosis(data['Daily Returns'],fisher=True)  
    
    n = len(data)
    
    std = ( (1 / (n-1)) * (1 + 0.5 * sr**2 - skew * sr + (kurtosis / 4) * sr**2))**0.5
    
    ratio = (sr - benchmark) / std
    prob_sharpe_ratio = scipy.stats.norm.cdf(ratio)
    return prob_sharpe_ratio 

# to check probabilistic sharpe ratio 

In [102]:
compute_prob_sharpe_ratio(aapl)

np.float64(1.0)